In [1]:
from bs4 import BeautifulSoup
import re
import requests
from os import popen
import urllib.request
from pathlib import Path
from collections import defaultdict
import time

# url = "Thermodynamics of Folding.html"
headers = {
    'Access-Control-Allow-Origin': '*',
    'Access-Control-Allow-Methods': 'GET',
    'Access-Control-Allow-Headers': 'Content-Type',
    'Access-Control-Max-Age': '3600',
    'User-Agent': 'Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:52.0) Gecko/20100101 Firefox/52.0'
    }

url = "http://www.unafold.org/results/14/24Nov03-14-10-10/"
mfold_url = 'http://www.unafold.org/'
storage_folder = '/home/dwarrel/Documents/data/Arenavirus/mfold_pages/GGV-S-IGR-presentation'
req = requests.get(url, headers)
soup = BeautifulSoup(req.content, 'html.parser')
gibs_to_kt = 1.689

In [4]:
def clean_dt_file(dt_file):
    result = [] 
    with open(dt_file, 'r') as file:
        lines = file.readlines()
        result.append(lines[0])
        result.append(lines[1].split('\t')[0])
        # lines[1] = lines.readlines()[1].split('\t')[0]
    result.append('\n')
    with open(dt_file, 'w') as file:
        file.writelines(result)
        
def db_to_lengths(db_file):
    with open(db_file, 'r') as file:
        db = file.readlines()[1].strip()
    i, j, initial_unpaired, paired, unpaired, unfolding, levels = 0, len(db)-1, 0, [], [], [], defaultdict(int)
    while db[i] == '.': initial_unpaired, i = initial_unpaired+1, i+1
    levels[0] = initial_unpaired
    nested = 0
    while i < len(db):
        if db[i] == '(':
            nested += 1
        while i < len(db) and (db[i] == '(' or (i+1 < len(db) and db[i] == '.' and db[i+1] == '(')):
            if db[i] == '(': 
                paired.append([i, -1, 1])
            if db[i] == '.': 
                paired[-1][2] += 1
            i += 1
        while db[i] == '.':
            paired[-1][2] += 1
            i += 1
        if db[i] == ')': # Closing time
            counter = 0
            last_opening  = paired.pop()
            counter += last_opening[2]+1 # count opposite bracket and dots plus current one
            i += 1 # We manually count the first one such that the hairpin head doesnt get counted as a large buldge
            while (i < len(db) and db[i] == ')') or (i+1 < len(db) and db[i] == '.' and db[i+1] == ')'):
                if db[i] == ')': 
                    if paired[-1][2] > 2: # Opposite had a buldge larger than 1 create new section  
                        break
                    # last_opening  = paired.pop()
                    counter += paired.pop()[2]+1 # count opposite bracket and dots plus current one
                if db[i] == '.': 
                    counter += 1 # just count the buldge
                i += 1
            # if nested-1 == 0 and len(paired):  # We had a nested 
            unfolding.append([counter, 1, last_opening[0], i-1])
            for unfold in unfolding:
                if last_opening[0] < unfold[2] and unfold[3] < i:
                    unfold[1] += 1
                
            nested -= 1
            counter = 0
            while i < len(db) and db[i] == '.':
                counter += 1
                i += 1
            if len(paired):
                paired[-1][2] += counter
            else:
                initial_unpaired += counter
    unfolding.append((initial_unpaired, 0, 0, len(db)))
    result = []
    for ufold in unfolding:
        result.append((*ufold, round(ufold[0]*.59, 1)))
    return result

def store_lengths(unfoldings, out_path):
    resulting_str = ''
    for idx, unfolding in enumerate(unfoldings):
        resulting_str += f'Structure: {idx}\ndepth\t'
        transposed = list(map(list, zip(*unfolding)))
        for depth in transposed[1]:
            resulting_str += f"{depth}\t"
        resulting_str = resulting_str[:-1] + '\nnm\t\t'
        for nm in transposed[4]:
            resulting_str += f"{nm}\t"
        resulting_str = resulting_str[:-1] + '\nbases\t'
        for length in transposed[0]:
            resulting_str += f"{length}\t"
        resulting_str = resulting_str[:-1] + '\npos\t'
        for pos in zip(transposed[2],transposed[3]):
            resulting_str += f"{pos}\t"
        resulting_str = resulting_str[:-1] + '\n\n'
    resulting_str = resulting_str[:-1]
    with open(out_path, 'w') as file:
        file.write(resulting_str)

def get_ctplot(url, num_structures, title, out_path):
    title_plus = '+'.join(title.split(" "))
    _,_,_,_,url_id, url_date, _ =url.split('/')
    struc_str = ''
    for i in range(num_structures):
        struc_str += f"&C={i+1}"
    get_url = f"http://www.unafold.org/cgi-bin/display-img-ct_boxplot.cgi?FILE_NAME={url_id}%2F{url_date}%2F1&IMG_RES=110&TITLE={title_plus}&OTYPE=jpg{struc_str}"
    contents = urllib.request.urlopen(get_url).read()
    box_soup = BeautifulSoup(contents)
    boxct_url = box_soup.find(attrs={"name": "CT_PLOT"})['src']
    output = popen(f"wget -O {out_path} '{boxct_url}' ")

def download_mfold_results(url, storage_folder):
    req = requests.get(url, headers)
    soup = BeautifulSoup(req.content, 'html.parser')
    title = soup.find(attrs={"class": "mt-2 page-title"}).get_text().split('on ')[1].split(' for')[0]
    p_group = soup.find_all(lambda tag:tag.name=='p' and 'Structure ' in tag.text)[0]
    # print(p_group)
    structure_id = 1
    unfoldings = []
    for p_element in p_group.prettify().split('<p>')[1:]:
        Path(f"{storage_folder}/thermo/").mkdir(parents=True, exist_ok=True)
        Path(f"{storage_folder}/pdfs/").mkdir(parents=True, exist_ok=True)
        Path(f"{storage_folder}/cts/").mkdir(parents=True, exist_ok=True)
        Path(f"{storage_folder}/dots/").mkdir(parents=True, exist_ok=True)
        if 'Structure ' in p_element:
            p_soup = BeautifulSoup(p_element, 'html.parser')
            output = popen(f"wget -O {storage_folder}/thermo/{structure_id}.html '{mfold_url}{p_soup.find_all('a', href=True)[1]['href']}' ")
            output = popen(f"wget -O {storage_folder}/pdfs/{structure_id}.pdf '{mfold_url}{p_soup.find_all('a', href=True)[3]['href']}' ")
            output = popen(f"wget -O {storage_folder}/cts/{structure_id}.ct '{mfold_url}{p_soup.find_all('a', href=True)[6]['href']}' ")
            output = popen(f"wget -O {storage_folder}/dots/{structure_id}.dt '{mfold_url}{p_soup.find_all('a', href=True)[7]['href']}' ")
            time.sleep(2)
            clean_dt_file(f"{storage_folder}/dots/{structure_id}.dt")
            unfoldings.append(db_to_lengths(f"{storage_folder}/dots/{structure_id}.dt"))
            structure_id += 1
    store_lengths(unfoldings, f"{storage_folder}/unfolds.txt")
    get_ctplot(url, len(unfoldings), title, f"{storage_folder}/box_ct.jpg")

download_mfold_results(url, storage_folder)

--2024-11-03 15:11:57--  http://www.unafold.org//cgi-bin/det.cgi?ID=24Nov03-14-10-10&COUNT=1
Resolving www.unafold.org (www.unafold.org)... --2024-11-03 15:11:57--  http://www.unafold.org//cgi-bin/ps2pdf.cgi?ID=24Nov03-14-10-10&COUNT=1
Resolving www.unafold.org (www.unafold.org)... --2024-11-03 15:11:57--  http://www.unafold.org//cgi-bin/ct.cgi?ID=24Nov03-14-10-10&COUNT=1
Resolving www.unafold.org (www.unafold.org)... 18.219.218.2718.219.218.27

Connecting to www.unafold.org (www.unafold.org)|18.219.218.27|:80... Connecting to www.unafold.org (www.unafold.org)|18.219.218.27|:80... 18.219.218.27
Connecting to www.unafold.org (www.unafold.org)|18.219.218.27|:80... --2024-11-03 15:11:57--  http://www.unafold.org//cgi-bin/ct2b.cgi?ID=24Nov03-14-10-10&COUNT=1
Resolving www.unafold.org (www.unafold.org)... 18.219.218.27
Connecting to www.unafold.org (www.unafold.org)|18.219.218.27|:80... connected.
HTTP request sent, awaiting response... connected.
connected.
HTTP request sent, awaiting resp

In [116]:
# contents = urllib.request.urlopen("http://www.unafold.org/cgi-bin/display-img-ct_boxplot.cgi?FILE_NAME=8%2F24Sep13-08-28-42%2F1&IMG_RES=110&TITLE=ADD4k+Dresden&OTYPE=jpg&C=1&C=2&C=3&C=4").read()
# print(contents)
# req = requests.get(url, headers)
soup = BeautifulSoup(req.content, 'html.parser')
title = soup.find(attrs={"class": "mt-2 page-title"}).get_text().split('on ')[1].split(' for')[0]
print(title)

ADD4k Dresden


In [123]:

# box_soup = BeautifulSoup(contents)
# box = box_soup.find(attrs={"name": "CT_PLOT"})['src']
get_ctplot(url, 4, title)

'http://www.unafold.org/results/8/24Sep13-08-21-27/1_boxct.jpg'

In [273]:


def db_to_lengths2(db_file):
    with open(db_file, 'r') as file:
        db = file.readlines()[1].strip()
    i, j, initial_unpaired, paired, unpaired, unfolding, levels = 0, len(db)-1, 0, [], [], [], defaultdict(int)
    while db[i] == '.': initial_unpaired, i = initial_unpaired+1, i+1
    levels[0] = initial_unpaired
    nested = 0
    while i < len(db):
        if db[i] == '(':
            nested += 1
        while i < len(db) and (db[i] == '(' or (i+1 < len(db) and db[i] == '.' and db[i+1] == '(')):
            if db[i] == '(': 
                paired.append([i, -1, 1])
            if db[i] == '.': 
                paired[-1][2] += 1
            i += 1
        while db[i] == '.':
            paired[-1][2] += 1
            i += 1
        if db[i] == ')': # Closing time
            counter = 0
            last_opening  = paired.pop()
            counter += last_opening[2]+1 # count opposite bracket and dots plus current one
            i += 1 # We manually count the first one such that the hairpin head doesnt get counted as a large buldge
            while (i < len(db) and db[i] == ')') or (i+1 < len(db) and db[i] == '.' and db[i+1] == ')'):
                if db[i] == ')': 
                    if paired[-1][2] > 2: # Opposite had a buldge larger than 1 create new section  
                        break
                    # last_opening  = paired.pop()
                    counter += paired.pop()[2]+1 # count opposite bracket and dots plus current one
                if db[i] == '.': 
                    counter += 1 # just count the buldge
                i += 1
            # if nested-1 == 0 and len(paired):  # We had a nested 
            unfolding.append([counter, 1, last_opening[0], i-1])
            for unfold in unfolding:
                if last_opening[0] < unfold[2] and unfold[3] < i:
                    unfold[1] += 1
                
            nested -= 1
            counter = 0
            while i < len(db) and db[i] == '.':
                counter += 1
                i += 1
            if len(paired):
                paired[-1][2] += counter
            else:
                initial_unpaired += counter
    unfolding.append((initial_unpaired, 0, 0, len(db)))
    result = []
    for ufold in unfolding:
        result.append((*ufold, round(ufold[0]*.59, 1)))
    return result
x = db_to_lengths2('/home/dwarrel/projects/temp-gits/draw_rna/example_files/mfold_pages/LMVC-L-IGR/ct2dot_LCMVL_RESTRAINED.txt')
print(x)
print([i[1] for i in x])
print([i[0] for i in x])
print([(i[2],i[3]) for i in x])
print([i[4] for i in x])
    

[(21, 1, 4, 20, 12.4), (32, 3, 65, 82, 18.9), (46, 3, 100, 128, 27.1), (20, 4, 155, 167, 11.8), (35, 3, 139, 185, 20.6), (46, 2, 50, 207, 27.1), (16, 1, 28, 216, 9.4), (1, 0, 0, 217, 0.6)]
[1, 3, 3, 4, 3, 2, 1, 0]
[21, 32, 46, 20, 35, 46, 16, 1]
[(4, 20), (65, 82), (100, 128), (155, 167), (139, 185), (50, 207), (28, 216), (0, 217)]
[12.4, 18.9, 27.1, 11.8, 20.6, 27.1, 9.4, 0.6]
